In [ ]:
from torch.utils.data import DataLoader
%load_ext autoreload
%autoreload 2

import torch
from torch import optim
from tqdm import tqdm
import time
from torchvision.datasets import CelebA
from matplotlib import pyplot as plt

from meles.frame_aggregation.pix2pix import Pix2PixGenerator, Pix2PixDiscriminator, Pix2Pix, pix2pix_transform

In [ ]:
# Defining Training Tools

num_epochs = 50
learning_rate = 2e-4
lambda_l1 = 100
display_interval = 10
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

generator = Pix2PixGenerator()
discriminator = Pix2PixDiscriminator()
pix2pix = Pix2Pix(generator, discriminator)

g_optimizer = optim.Adam(generator.parameters(), lr=learning_rate, betas=(0.5, 0.999))
d_optimizer = optim.Adam(discriminator.parameters(), lr=learning_rate, betas=(0.5, 0.999))

def train_pix2pix(pix2pix_model, train_loader, val_loader, epochs, lambda_l1, display_interval):

    start_time = time.time()
    train_g_losses, train_d_losses = [], []
    val_g_losses, val_d_losses = [], []

    for epoch in range(epochs):

        # training
        pix2pix_model.train()
        train_g_loss, train_d_loss = 0.0, 0.0

        for idx, (segmented_images, real_images) in enumerate(tqdm(train_loader, desc=f"Training {epoch+1}/{epochs}")):
            segmented_images = segmented_images.to(device)
            real_images = real_images.to(device)

            # train model discriminator
            d_optimizer.zero_grad()
            fake_images = pix2pix_model.generator(segmented_images)
            real_output = pix2pix_model.discriminator(segmented_images, real_images)
            fake_output = pix2pix_model.discriminator(segmented_images, fake_images)
            d_loss = pix2pix_model.discriminator_loss(real_output, fake_output)
            d_loss.backward()
            d_optimizer.step()

            # train model generator
            g_optimizer.zero_grad()
            fake_images = pix2pix_model.generator(segmented_images)
            fake_output = pix2pix_model.discriminator(segmented_images, fake_images)
            g_loss = pix2pix_model.generator_loss(fake_output, fake_images, real_images, lambda_l1)
            g_loss.backward()
            g_optimizer.step()

            train_d_loss += d_loss.item()
            train_g_loss += g_loss.item()

        # average train loss per epoch
        train_d_losses.append(train_d_loss / len(train_loader))
        train_g_losses.append(train_g_loss / len(train_loader))

        # validating
        pix2pix_model.eval()
        val_g_loss, val_d_loss = 0.0, 0.0

        with torch.no_grad():
            for idx, (segmented_images, real_images) in enumerate(tqdm(val_loader, desc=f"Validating {epoch+1}/{epochs}")):
                segmented_images = segmented_images.to(device)
                real_images = real_images.to(device)

                fake_images = pix2pix_model.generator(segmented_images)
                real_output = pix2pix_model.discriminator(segmented_images, real_images)
                fake_output = pix2pix_model.discriminator(segmented_images, fake_images)

                d_loss = pix2pix_model.discriminator_loss(real_output, fake_output)
                g_loss = pix2pix_model.generator_loss(fake_output, fake_images, real_images, lambda_l1)

                val_d_loss += d_loss.item()
                val_g_loss += g_loss.item()

        # average val loss per epoch
        val_d_losses.append(val_d_loss / len(val_loader))
        val_g_losses.append(val_g_loss / len(val_loader))

        print(f"Epoch {epoch + 1}/{epochs} | G Train Loss: {train_g_losses[-1]:.4f} | D Train Loss: {train_d_losses[-1]:.4f} | G Val Loss: {val_g_losses[-1]:.4f} | D Val Loss: {val_d_losses[-1]:.4f}")

        # display sample results after each interval
        if (epoch + 1) % display_interval == 0 or (epoch + 1) == epochs:
            display_generated_samples(pix2pix_model, train_loader, num_samples=3)

    print(f"Training Completed in {time.time() - start_time:.2f} seconds.")
    return train_g_losses, train_d_losses, val_g_losses, val_d_losses


def display_generated_samples(model, loader, num_samples=10, title="Generated Samples"):

    model.eval()
    with torch.no_grad():
        segmented_images, real_images = next(iter(loader))
        segmented_images = segmented_images[:num_samples].to(device)
        real_images = real_images[:num_samples].to(device)

        fake_images = model.generator(segmented_images)

        fig, axes = plt.subplots(num_samples, 3, figsize=(15, 5 * num_samples))
        fig.suptitle(title, fontsize=16)

        for i in range(num_samples):
            axes[i, 0].imshow((segmented_images[i].permute(1, 2, 0).cpu().numpy() + 1) / 2)         # denormalizing
            axes[i, 0].set_title("Input Image")
            axes[i, 0].axis('off')
            axes[i, 1].imshow((real_images[i].permute(1, 2, 0).cpu().numpy() + 1) / 2)              # denormalizing
            axes[i, 1].set_title("Target Image")
            axes[i, 1].axis('off')
            axes[i, 2].imshow((fake_images[i].permute(1, 2, 0).cpu().numpy() + 1) / 2)              # denormalizing
            axes[i, 2].set_title("Generated Image")
            axes[i, 2].axis('off')

        plt.tight_layout()
        plt.show()

In [ ]:
SIZE = 256
batch_size = 32
epochs = 50

def anonymize(foo):
    # TODO implement me!
    return foo

# TODO update download dir
train_dataset = CelebA(".data", split="train", download=True, transform=pix2pix_transform ,target_transform=anonymize)
test_dataset = CelebA(".data", split="test", download=True, transform=pix2pix_transform, target_transform=anonymize)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, persistent_workers=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, persistent_workers=True)

train_pix2pix(pix2pix, train_dataset, test_dataset, epochs, lambda_l1, display_interval)